# **Concurrency:**
#### Ability to deal with multiple tasks at the same time (tasks may start, run, and complete overlapping in time).


# **ASYNCIO:**
- asyncio is a Python library that provides single-threaded, cooperative concurrency using an event loop.
- Tasks run within one thread but voluntarily give up control when they reach an await (like waiting for I/O), so other tasks can run in the meantime.<br>

**Best for:** Many I/O-bound operations (e.g., API calls, database queries, file/network I/O) where tasks spend time waiting.<br>

**Analogy:** One waiter serving many tables by moving between them while food is being prepared.

## **Synchronous version (blocking)**
- **Definition**: Operations execute one after another.

- Matlab, ek kaam complete hone ke baad hi dusra kaam start hota hai.
- Agar koi operation slow hai (jaise network request ya file read), to program us waqt ruk jata hai aur wait karta hai.

In [ ]:
import requests
import time
import asyncio
import aiohttp

In [13]:
def api_1():
    # Simulate a delay of 5 seconds
    time.sleep(5)
    response = requests.get("https://jsonplaceholder.typicode.com/posts")
    print("posts-->",response.json())

def api_2():
    # Simulate a delay of 1 second
    time.sleep(1)
    response = requests.get("https://jsonplaceholder.typicode.com/users")
    print("users-->",response.json())

In [14]:
# Calling functions synchronously
# api_1() will complete entirely before api_2() starts
# api_2() will then run
# This is blocking execution
api_1()
api_2()
# abhi yaha 1 kam complete hony k baad dusra chla

posts--> [{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}, {'userId': 1, 'id': 2, 'title': 'qui est esse', 'body': 'est rerum tempore vitae\nsequi sint nihil reprehenderit dolor beatae ea dolores neque\nfugiat blanditiis voluptate porro vel nihil molestiae ut reiciendis\nqui aperiam non debitis possimus qui neque nisi nulla'}, {'userId': 1, 'id': 3, 'title': 'ea molestias quasi exercitationem repellat qui ipsa sit aut', 'body': 'et iusto sed quo iure\nvoluptatem occaecati omnis eligendi aut ad\nvoluptatem doloribus vel accusantium quis pariatur\nmolestiae porro eius odio et labore et velit aut'}, {'userId': 1, 'id': 4, 'title': 'eum et est occaecati', 'body': 'ullam et saepe reiciendis voluptatem adipisci\nsit amet autem assumenda provident rerum culpa\nq

## **Asynchronous version (non-blocking)**
**Definition:** Operations can start and pause independently, allowing other tasks to run while waiting.

- Matlab, program ek task ke complete hone ka wait nahi karta, wo background me chal sakta hai aur dusre tasks bhi execute ho sakte hain.

- Python me async ka use async def aur await ke saath hota hai.

In [15]:
async def api_1():
    # Simulate a delay of 5 seconds asynchronously
    # This does not block the event loop, other tasks can run during this time
    await asyncio.sleep(5)

    # Create an asynchronous HTTP session
    # aiohttp.ClientSession manages connections efficiently
    async with aiohttp.ClientSession() as session:
        # Send a GET request asynchronously
        async with session.get("https://jsonplaceholder.typicode.com/posts") as response:
            # Await the JSON response (async operation)
            # 'response.json()' is a coroutine, we must await it
            # await pauses this line until the server sends full JSON data
            data = await response.json()
            print("posts-->",data)
    
async def api_2():
    await asyncio.sleep(2)
    async with aiohttp.ClientSession() as session:
        async with session.get("https://jsonplaceholder.typicode.com/users") as response:
            data = await response.json()
            print("users-->",data)

In [16]:
# To run asynchronous functions, we need an event loop
# asyncio.run gathers the coroutines and executes them concurrently
async def main():
    # asyncio.gather runs multiple coroutines concurrently
    await asyncio.gather(api_1(),api_2())
    
# asyncio.run(main())  # for normal .py file

# Run the main async function
# This will start both api_1 and api_2 at the same time
await main()

users--> [{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'address': {'street': 'Kulas Light', 'suite': 'Apt. 556', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'geo': {'lat': '-37.3159', 'lng': '81.1496'}}, 'phone': '1-770-736-8031 x56442', 'website': 'hildegard.org', 'company': {'name': 'Romaguera-Crona', 'catchPhrase': 'Multi-layered client-server neural-net', 'bs': 'harness real-time e-markets'}}, {'id': 2, 'name': 'Ervin Howell', 'username': 'Antonette', 'email': 'Shanna@melissa.tv', 'address': {'street': 'Victor Plains', 'suite': 'Suite 879', 'city': 'Wisokyburgh', 'zipcode': '90566-7771', 'geo': {'lat': '-43.9509', 'lng': '-34.4618'}}, 'phone': '010-692-6593 x09125', 'website': 'anastasia.net', 'company': {'name': 'Deckow-Crist', 'catchPhrase': 'Proactive didactic contingency', 'bs': 'synergize scalable supply-chains'}}, {'id': 3, 'name': 'Clementine Bauch', 'username': 'Samantha', 'email': 'Nathan@yesenia.net', 'address': {'street': 'Doug

In [17]:
'''
asyncio.sleep(5) aur asyncio.sleep(2)

Ye actual CPU ko block nahi karte.

Matlab jab api_1 5 second ka wait kar raha hai, event loop ko free chhod deta hai taake api_2 ya koi aur coroutine chal sake.

asyncio.gather(api_1(), api_2())

Ye dono coroutines ko ek saath schedule karta hai, logically concurrent chal rahe hain.

Concurrently ka matlab: dono tasks same time pe start ho gaye, lekin CPU ek hi time me sirf ek kaam execute kar sakta hai (agar single-core hai).

Step by step execution:

Time 0: api_1 start → 5s wait, api_2 start → 2s wait

Time 2s: api_2 ka sleep khatam, HTTP request send hoti hai → response receive hota hai → print "users--> ..."

Time 5s: api_1 ka sleep khatam, HTTP request send hoti hai → response receive hota hai → print "posts--> ..."

Key points:

Dono coroutines simultaneously start ho rahe hain, is liye hum kehte hain concurrent execution.

Lekin, actual execution CPU ke upar sequential lag sakti hai, kyunki ek core ek time me ek instruction execute karta hai.

Jab hum I/O-bound tasks (jaise HTTP requests, sleep) ke liye async/await use karte hain, tab ye highly efficient hai kyunki CPU ka time waste nahi hota.
'''

'\nasyncio.sleep(5) aur asyncio.sleep(2)\n\nYe actual CPU ko block nahi karte.\n\nMatlab jab api_1 5 second ka wait kar raha hai, event loop ko free chhod deta hai taake api_2 ya koi aur coroutine chal sake.\n\nasyncio.gather(api_1(), api_2())\n\nYe dono coroutines ko ek saath schedule karta hai, logically concurrent chal rahe hain.\n\nConcurrently ka matlab: dono tasks same time pe start ho gaye, lekin CPU ek hi time me sirf ek kaam execute kar sakta hai (agar single-core hai).\n\nStep by step execution:\n\nTime 0: api_1 start → 5s wait, api_2 start → 2s wait\n\nTime 2s: api_2 ka sleep khatam, HTTP request send hoti hai → response receive hota hai → print "users--> ..."\n\nTime 5s: api_1 ka sleep khatam, HTTP request send hoti hai → response receive hota hai → print "posts--> ..."\n\nKey points:\n\nDono coroutines simultaneously start ho rahe hain, is liye hum kehte hain concurrent execution.\n\nLekin, actual execution CPU ke upar sequential lag sakti hai, kyunki ek core ek time me ek

In [ ]:
async def func1():
    """
    func1 simulates a task that starts, waits asynchronously for 3 seconds,
    and then prints numbers 0 to 4.
    
    Key points:
    - 'await asyncio.sleep(3)' frees the event loop for other tasks while waiting.
    - After 3 seconds, func1 resumes printing numbers.
    """
    print("Function1 started")    
    await asyncio.sleep(1)
    for i in range(5):
        print(f"Func1: {i}")
    
async def func2():
    """
    func2 is a CPU-bound task that prints numbers 0 to 9.
    
    Key points:
    - There is NO 'await' here, so this loop is synchronous.
    - While func2 is running, the event loop CANNOT switch to other tasks.
    - This blocks func1 from resuming if it is waiting to print after await.
    """
    for i in range(1000):
        print(f"Func2: {i}")


In [ ]:
async def main():
    '''
    Behavior:
    - func1 starts and hits 'await asyncio.sleep(3)', releasing control to the event loop.
    - func2 runs immediately because func1 is waiting.
    - After 3 seconds, func1 resumes and prints its loop.
    - Demonstrates concurrency in asyncio:
      multiple tasks progress independently where 'await' is used.
    '''
    await asyncio.gather(func1(),func2())
    
# asyncio.run(main())  # for normal .py file
await main()

Function1 started
Func2: 0
Func2: 1
Func2: 2
Func2: 3
Func2: 4
Func2: 5
Func2: 6
Func2: 7
Func2: 8
Func2: 9
Func2: 10
Func2: 11
Func2: 12
Func2: 13
Func2: 14
Func2: 15
Func2: 16
Func2: 17
Func2: 18
Func2: 19
Func2: 20
Func2: 21
Func2: 22
Func2: 23
Func2: 24
Func2: 25
Func2: 26
Func2: 27
Func2: 28
Func2: 29
Func2: 30
Func2: 31
Func2: 32
Func2: 33
Func2: 34
Func2: 35
Func2: 36
Func2: 37
Func2: 38
Func2: 39
Func2: 40
Func2: 41
Func2: 42
Func2: 43
Func2: 44
Func2: 45
Func2: 46
Func2: 47
Func2: 48
Func2: 49
Func2: 50
Func2: 51
Func2: 52
Func2: 53
Func2: 54
Func2: 55
Func2: 56
Func2: 57
Func2: 58
Func2: 59
Func2: 60
Func2: 61
Func2: 62
Func2: 63
Func2: 64
Func2: 65
Func2: 66
Func2: 67
Func2: 68
Func2: 69
Func2: 70
Func2: 71
Func2: 72
Func2: 73
Func2: 74
Func2: 75
Func2: 76
Func2: 77
Func2: 78
Func2: 79
Func2: 80
Func2: 81
Func2: 82
Func2: 83
Func2: 84
Func2: 85
Func2: 86
Func2: 87
Func2: 88
Func2: 89
Func2: 90
Func2: 91
Func2: 92
Func2: 93
Func2: 94
Func2: 95
Func2: 96
Func2: 97
Func2: 98
Fu